# Compressor Health Clustering

## 1. K-Means Clustering and Chronological Evaluation

This notebook consumes the 5-minute feature table created by notebook 02. It removes low-coverage windows from the initial model matrix, reserves the latest 20% of eligible windows as a chronological holdout, and fits imputation and robust scaling on training data only. It evaluates candidate cluster counts with inertia, sampled silhouette, and cluster sizes, then checks initialization stability and assigns holdout windows without refitting.

`PROVISIONAL_K` is set to 4 as an editable starting point. Review the candidate diagnostics before accepting that value. Cluster numbers are arbitrary operating-behavior groups; they are not health grades or fault predictions. The notebook is intentionally unexecuted.

## 2. Import Libraries and Locate the Processed Features

Keep the same dark charcoal chart style as the previous notebooks. The processed CSV files are generated by running notebook 02.

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

DATA_RELATIVE_PATH = Path("data") / "processed" / "metropt_window_features_5min.csv"
PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / DATA_RELATIVE_PATH).is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Processed features were not found. Run notebook 02 and save its outputs first."
    )
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_PATH = PROCESSED_DIR / "metropt_window_features_5min.csv"
METADATA_PATH = PROCESSED_DIR / "metropt_window_metadata_5min.csv"

PALETTE = {
    "blue": "#72B7F2", "teal": "#4FD1C5", "amber": "#F6C177",
    "red": "#F28B82", "gray": "#AAB6C2", "figure_bg": "#151B23",
    "axes_bg": "#202833", "text": "#E6EDF3", "muted_text": "#B8C4D0",
    "grid": "#3B4653",
}
sns.set_theme(style="darkgrid", context="notebook", font="DejaVu Sans", font_scale=1.0)
plt.rcParams.update({
    "figure.figsize": (11, 5), "figure.dpi": 120,
    "figure.facecolor": PALETTE["figure_bg"], "savefig.facecolor": PALETTE["figure_bg"],
    "axes.facecolor": PALETTE["axes_bg"], "axes.edgecolor": PALETTE["grid"],
    "axes.labelcolor": PALETTE["text"], "axes.titlecolor": PALETTE["text"],
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": PALETTE["muted_text"], "ytick.color": PALETTE["muted_text"],
    "text.color": PALETTE["text"], "grid.color": PALETTE["grid"], "grid.alpha": 0.55,
})
print(f"Feature file: {FEATURE_PATH}")
print(f"Metadata file: {METADATA_PATH}")

## 3. Load, Join, and Validate the Feature Tables

Join by the segment and window identifiers. Timestamps and coverage remain metadata and will not be passed to K-Means.

In [ ]:
FEATURE_KEYS = ["segment_id", "window_id"]
features = pd.read_csv(FEATURE_PATH)
metadata = pd.read_csv(METADATA_PATH, parse_dates=["window_start", "window_end"])

for name, table in [("features", features), ("metadata", metadata)]:
    missing_keys = sorted(set(FEATURE_KEYS) - set(table.columns))
    if missing_keys:
        raise ValueError(f"{name} table is missing join keys: {missing_keys}")

data = metadata.merge(
    features, on=FEATURE_KEYS, how="inner", validate="one_to_one",
    suffixes=("_metadata", ""),
).sort_values(["window_start", "segment_id", "window_id"]).reset_index(drop=True)

if len(data) != len(features) or len(data) != len(metadata):
    raise ValueError("Feature and metadata tables do not match one-to-one.")
FEATURE_COLUMNS = [column for column in features.columns if column not in FEATURE_KEYS]
if not FEATURE_COLUMNS:
    raise ValueError("No model features found in the feature CSV.")
if not all(pd.api.types.is_numeric_dtype(data[c]) for c in FEATURE_COLUMNS):
    raise TypeError("Every clustering input must be numeric.")

print(f"Joined windows: {len(data):,} | Candidate numeric features: {len(FEATURE_COLUMNS)}")
print(f"Time coverage: {data['window_start'].min()} to {data['window_end'].max()}")
display(data[[*FEATURE_KEYS, "window_start", "coverage_fraction", *FEATURE_COLUMNS[:5]]].head())

## 4. Define Eligible Windows and a Chronological Holdout

Exclude windows below the 80% coverage threshold from this first model fit; keep them in the source files for audit. Split eligible windows chronologically so later observations cannot influence preprocessing or model fitting.

In [ ]:
MIN_COVERAGE = 0.80
eligible = data.loc[data["coverage_fraction"] >= MIN_COVERAGE].copy()
eligible = eligible.sort_values(["window_start", "segment_id", "window_id"]).reset_index(drop=True)
if len(eligible) < 100:
    raise ValueError("Too few eligible windows for a meaningful chronological train/test split.")

split_index = int(np.floor(0.80 * len(eligible)))
train = eligible.iloc[:split_index].copy()
test = eligible.iloc[split_index:].copy()
if train.empty or test.empty:
    raise ValueError("Chronological split produced an empty partition.")

X_train_raw = train[FEATURE_COLUMNS]
X_test_raw = test[FEATURE_COLUMNS]
print(f"Eligible windows: {len(eligible):,} / {len(data):,}")
print(f"Training windows: {len(train):,} ({train['window_start'].min()} to {train['window_end'].max()})")
print(f"Holdout windows: {len(test):,} ({test['window_start'].min()} to {test['window_end'].max()})")
print(f"Holdout begins after the last training window: {train['window_end'].max() < test['window_start'].min()}")

## 5. Fit Training-Only Imputation and Scaling

A few one-sample windows have undefined standard deviations. Median imputation handles those values. Robust scaling reduces the influence of skewed ranges while keeping every learned transformation fitted on the training period only. K-Means remains sensitive to extreme values even after robust scaling, so sensor profiles must still be reviewed.

In [ ]:
imputer = SimpleImputer(strategy="median")
scaler = RobustScaler(quantile_range=(25.0, 75.0))
X_train_imputed = imputer.fit_transform(X_train_raw)
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_imputed = imputer.transform(X_test_raw)
X_test_scaled = scaler.transform(X_test_imputed)

print(f"Scaled training matrix: {X_train_scaled.shape}")
print(f"Scaled holdout matrix: {X_test_scaled.shape}")
print(f"Missing train values before imputation: {int(X_train_raw.isna().sum().sum()):,}")
print(f"Missing holdout values before imputation: {int(X_test_raw.isna().sum().sum()):,}")

## 6. Compare Candidate Cluster Counts

Fit K-Means for k=2 through 8 on the training period. Inertia is measured on all training windows. Silhouette is calculated on the same reproducible training sample for each candidate to keep the computation manageable. Cluster balance is reported alongside both metrics; no single score selects k automatically.

In [ ]:
CANDIDATE_K = range(2, 9)
SEED = 42
N_INIT = 20
SILHOUETTE_SAMPLE_SIZE = min(3_000, len(X_train_scaled))
rng = np.random.default_rng(SEED)
silhouette_indices = np.sort(
    rng.choice(len(X_train_scaled), size=SILHOUETTE_SAMPLE_SIZE, replace=False)
)

candidate_models = {}
metric_rows = []
for k in CANDIDATE_K:
    model = KMeans(n_clusters=k, init="k-means++", n_init=N_INIT, random_state=SEED)
    model.fit(X_train_scaled)
    sample_labels = model.labels_[silhouette_indices]
    if np.unique(sample_labels).size < 2:
        sample_silhouette = np.nan
    else:
        sample_silhouette = silhouette_score(
            X_train_scaled[silhouette_indices], sample_labels, metric="euclidean"
        )
    counts = np.bincount(model.labels_, minlength=k)
    candidate_models[k] = model
    metric_rows.append({
        "k": k,
        "training_inertia": model.inertia_,
        "training_sample_silhouette": sample_silhouette,
        "smallest_cluster_windows": int(counts.min()),
        "smallest_cluster_percent": float(counts.min() / counts.sum() * 100),
        "largest_cluster_percent": float(counts.max() / counts.sum() * 100),
    })

candidate_metrics = pd.DataFrame(metric_rows)
display(candidate_metrics.round(4))

## 7. Visualize the Model Diagnostics

The elbow curve shows diminishing returns in within-cluster dispersion; the silhouette curve summarizes separation on the training sample. The smallest-cluster share helps identify solutions that isolate only a tiny fraction of windows.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.lineplot(data=candidate_metrics, x="k", y="training_inertia", marker="o",
             color=PALETTE["blue"], ax=axes[0])
axes[0].set(title="Training inertia by cluster count", xlabel="Number of clusters (k)", ylabel="Inertia")
sns.lineplot(data=candidate_metrics, x="k", y="training_sample_silhouette", marker="o",
             color=PALETTE["teal"], ax=axes[1])
axes[1].set(title="Sampled training silhouette by cluster count",
            xlabel="Number of clusters (k)", ylabel="Silhouette score")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=candidate_metrics, x="k", y="smallest_cluster_percent",
            color=PALETTE["amber"], ax=ax)
ax.set(title="Smallest cluster share by candidate k", xlabel="Number of clusters (k)",
       ylabel="Smallest cluster (% of training windows)")
fig.tight_layout()
plt.show()

## 8. Review Initialization Stability for the Provisional k

Compare two independently initialized fits on the same training matrix using adjusted Rand index (ARI) on a fixed audit sample. ARI near 1 means the two fits assign the sampled windows similarly; a lower value signals sensitivity to initialization. This is a stability check, not a criterion to use alone.

In [ ]:
PROVISIONAL_K = 4  # Revisit after reviewing the candidate metrics, cluster sizes, and profiles.
if PROVISIONAL_K not in CANDIDATE_K:
    raise ValueError(f"PROVISIONAL_K must be one of {list(CANDIDATE_K)}")

STABILITY_SAMPLE_SIZE = min(5_000, len(X_train_scaled))
stability_indices = np.sort(
    np.random.default_rng(SEED + 1).choice(
        len(X_train_scaled), size=STABILITY_SAMPLE_SIZE, replace=False
    )
)
stability_a = KMeans(n_clusters=PROVISIONAL_K, n_init=N_INIT, random_state=SEED + 1)
stability_b = KMeans(n_clusters=PROVISIONAL_K, n_init=N_INIT, random_state=SEED + 2)
stability_a.fit(X_train_scaled)
stability_b.fit(X_train_scaled)
stability_ari = adjusted_rand_score(
    stability_a.labels_[stability_indices], stability_b.labels_[stability_indices]
)
print(f"ARI across two initializations at k={PROVISIONAL_K}: {stability_ari:.4f}")

## 9. Fit the Provisional Model and Evaluate the Holdout

Fit the reusable pipeline on the training period only. Then assign labels to the later holdout period without refitting. Holdout silhouette is descriptive and should be considered together with cluster-size shifts and stability.

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler(quantile_range=(25.0, 75.0))),
    ("kmeans", KMeans(n_clusters=PROVISIONAL_K, init="k-means++",
                      n_init=N_INIT, random_state=SEED)),
])
pipeline.fit(X_train_raw)
train_labels = pipeline.predict(X_train_raw)
test_labels = pipeline.predict(X_test_raw)

def sampled_silhouette(X, labels, max_samples=3_000, seed=42):
    """Compute silhouette on a reproducible sample, returning NaN if undefined."""
    if len(labels) < 3 or np.unique(labels).size < 2:
        return np.nan
    size = min(max_samples, len(labels))
    indices = np.sort(np.random.default_rng(seed).choice(len(labels), size=size, replace=False))
    sampled_labels = labels[indices]
    if np.unique(sampled_labels).size < 2 or np.unique(sampled_labels).size >= size:
        return np.nan
    transformed = pipeline.named_steps["scaler"].transform(
        pipeline.named_steps["imputer"].transform(X.iloc[indices])
    )
    return silhouette_score(transformed, sampled_labels, metric="euclidean")

train_silhouette = sampled_silhouette(X_train_raw, train_labels, seed=SEED + 3)
test_silhouette = sampled_silhouette(X_test_raw, test_labels, seed=SEED + 4)
print(f"Training sampled silhouette: {train_silhouette:.4f}")
print(f"Holdout sampled silhouette: {test_silhouette:.4f}")

train_cluster_sizes = pd.Series(train_labels, name="cluster").value_counts().sort_index()
test_cluster_sizes = pd.Series(test_labels, name="cluster").value_counts().sort_index()
cluster_sizes = pd.concat([train_cluster_sizes.rename("train_windows"),
                           test_cluster_sizes.rename("holdout_windows")], axis=1).fillna(0).astype(int)
display(cluster_sizes)

## 10. Inspect Cluster Sizes and a PCA Projection

Cluster IDs are arbitrary. This projection is a visual summary of the training data after training-only preprocessing; PCA can distort distances, so use it to inspect broad structure rather than to validate the clusters by itself.

In [ ]:
train_scaled_for_pca = pipeline.named_steps["scaler"].transform(
    pipeline.named_steps["imputer"].transform(X_train_raw)
)
pca = PCA(n_components=2, random_state=SEED)
train_pca = pca.fit_transform(train_scaled_for_pca)
pca_view = pd.DataFrame({"PC1": train_pca[:, 0], "PC2": train_pca[:, 1],
                         "cluster": train_labels.astype(str)})
pca_view = pca_view.sample(n=min(5_000, len(pca_view)), random_state=SEED)
print(f"Variance explained by first two PCs: {pca.explained_variance_ratio_.sum():.1%}")

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=pca_view, x="PC1", y="PC2", hue="cluster",
                palette=sns.color_palette("colorblind", n_colors=PROVISIONAL_K),
                alpha=0.55, s=18, linewidth=0, ax=ax)
ax.set_title(f"Training windows in PCA space (provisional k={PROVISIONAL_K})")
ax.legend(title="Cluster", frameon=False, labelcolor=PALETTE["muted_text"])
fig.tight_layout()
plt.show()

size_plot = cluster_sizes.reset_index().melt(id_vars="cluster", var_name="partition", value_name="windows")
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=size_plot, x="cluster", y="windows", hue="partition",
            palette=[PALETTE["blue"], PALETTE["amber"]], ax=ax)
ax.set(title="Cluster sizes in training and chronological holdout",
       xlabel="Arbitrary cluster ID", ylabel="Windows")
ax.legend(title="Partition", frameon=False)
fig.tight_layout()
plt.show()

## 11. Review Sensor Profiles Without Assigning Health Labels

Summarize medians of selected raw window features by training cluster. The table helps describe operating behavior and identify duplicated or uninformative dimensions. Do not label a cluster as healthy or faulty from these summaries alone.

In [ ]:
profile_columns = [
    "TP2__mean", "TP3__mean", "Reservoirs__mean", "H1__mean",
    "DV_pressure__mean", "Oil_temperature__mean", "Motor_current__mean",
    "COMP__fraction_value_1", "DV_eletric__fraction_value_1",
    "core_state_transitions",
]
profile_columns = [column for column in profile_columns if column in FEATURE_COLUMNS]
train_profiles = train[FEATURE_COLUMNS].copy()
train_profiles["cluster"] = train_labels
cluster_profiles = train_profiles.groupby("cluster")[profile_columns].median().round(3)
display(cluster_profiles)

## 12. Save the Pipeline and Window Assignments

Save the full preprocessing-plus-clustering pipeline, its feature schema, split dates, and provisional k. Also save chronological cluster assignments for notebook 04. The pipeline is fitted only on the training period; the holdout labels are produced by prediction.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / "kmeans_pipeline_5min.joblib"
metadata_path = MODEL_DIR / "kmeans_metadata_5min.json"
assignments_path = PROCESSED_DIR / "metropt_window_clusters_5min.csv"

joblib.dump(pipeline, model_path)
model_metadata = {
    "algorithm": "KMeans",
    "provisional_k": int(PROVISIONAL_K),
    "window": "5min",
    "minimum_coverage": MIN_COVERAGE,
    "feature_columns": FEATURE_COLUMNS,
    "training_start": str(train["window_start"].min()),
    "training_end": str(train["window_end"].max()),
    "holdout_start": str(test["window_start"].min()),
    "holdout_end": str(test["window_end"].max()),
    "random_state": SEED,
    "n_init": N_INIT,
    "stability_ari": float(stability_ari),
    "training_sample_silhouette": float(train_silhouette),
    "holdout_sample_silhouette": float(test_silhouette),
    "limitations": "Unsupervised operating-behavior clusters; not fault labels or failure probabilities.",
}
metadata_path.write_text(json.dumps(model_metadata, indent=2), encoding="utf-8")

assignments = pd.concat([train, test], ignore_index=True)
assignments["cluster"] = np.concatenate([train_labels, test_labels])
assignments["partition"] = ["train"] * len(train) + ["holdout"] * len(test)
assignment_columns = [*FEATURE_KEYS, "window_start", "window_end",
                      "coverage_fraction", "partition", "cluster"]
assignments[assignment_columns].to_csv(assignments_path, index=False)
print(f"Saved pipeline: {model_path}")
print(f"Saved metadata: {metadata_path}")
print(f"Saved assignments: {assignments_path}")

## 13. Findings and Limitations

After running and reviewing the diagnostics, replace this checklist with the selected k and evidence:

- **Cluster count:** explain the trade-off across inertia, sampled silhouette, smallest-cluster size, and interpretability.
- **Stability:** report the repeated-initialization ARI and whether it supports the chosen solution.
- **Holdout behavior:** compare chronological holdout silhouette and cluster proportions with training; describe material shifts without calling them faults.
- **Cluster profiles:** summarize the sensor/state features that distinguish groups and note redundancy such as TP3/Reservoirs.

### Limitations

K-Means assumes Euclidean, roughly spherical clusters and can be sensitive to skew and extreme values. Robust scaling does not eliminate that sensitivity. Silhouette is estimated on a sample, cluster IDs have no inherent order, and the chronological holdout is from the same compressor and operating history. No event labels are used to fit the clusters. A high or low metric alone does not establish an equipment fault or prospective warning.